# 02. 情节记忆与 Reflexion

**Reflexion** 的核心思想：Agent 不需要更新模型权重就能「学习」——它通过文本反思从失败中提取教训，存储在情节记忆中，在后续尝试中应用。

Reflexion 循环：
```
Attempt → Fail → Reflect (生成文字反思) → Store in Episodic Memory → Retry (应用历史反思)
```

> **检查点**：能区分 Reflexion 与传统 RL 在学习机制上的本质区别。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
from dataclasses import dataclass, field

@dataclass
class Episode:
    task: str; action: str; outcome: str; success: bool; reflection: str = ""

class ReflexionLoop:
    def __init__(self):
        self.episodes = []
        self.lessons = []
    
    def attempt(self, task: str, attempt_num: int) -> tuple[str, bool]:
        """模拟一次尝试——策略随反思改进"""
        if "coding" in task and attempt_num == 1:
            return ("wrote code without tests", False)
        if "coding" in task and attempt_num >= 2:
            if any("test" in l for l in self.lessons):
                return ("wrote tests first, then implemented", True)
        if "math" in task and attempt_num == 1:
            return ("guessed 42", False)
        if "math" in task and attempt_num >= 2:
            if any("verify" in l for l in self.lessons):
                return ("computed step by step, verified", True)
        return ("unknown approach", False)
    
    def reflect(self, ep: Episode) -> str:
        if "without tests" in ep.action:
            r = "Always write tests before implementing. TDD prevents regressions."
        elif "guessed" in ep.action:
            r = "Never guess. Compute methodically and verify the result."
        else:
            r = "Review the approach and try a systematic method."
        self.lessons.append(r)
        return r
    
    def run(self, task: str, max_attempts: int = 3):
        print(f"Task: {task}\n")
        for n in range(1, max_attempts + 1):
            action, ok = self.attempt(task, n)
            ep = Episode(task, action, "success" if ok else "failed", ok)
            status = "✓" if ok else "✗"
            print(f"  Attempt {n}: {action} [{status}]")
            if ok:
                self.episodes.append(ep)
                print(f"  Solved! Lessons used: {len(self.lessons)}")
                return True
            ep.reflection = self.reflect(ep)
            self.episodes.append(ep)
            print(f"  → Reflection: {ep.reflection}\n")
        return False

ReflexionLoop().run("solve math problem", 3)

Task: solve math problem

  Attempt 1: guessed 42 [✗]
  → Reflection: Never guess. Compute methodically and verify the result.

  Attempt 2: computed step by step, verified [✓]
  Solved! Lessons used: 1


True

## 情节记忆的数据结构

一个好的情节记忆系统应该存储：

```python
{
  "task": "deploy to production",
  "context": {"env": "k8s", "version": "2.1.0"},
  "actions": ["kubectl apply -f deploy.yaml"],
  "outcome": "CrashLoopBackOff",
  "root_cause": "Missing DATABASE_URL env var",
  "lesson": "Validate all required env vars before deployment",
  "timestamp": "2025-07-23T10:00:00Z"
}
```

> **检查点**：能设计一个情节记忆的 schema，包含足够的字段来支持未来的检索和反思。

## 真实 API 实验

真实 Reflexion：让 LLM 解题 → 故意评判失败 → LLM 写反思 → 带反思重试，观察正确率变化。


In [3]:
# 真实 Reflexion 循环（一轮）
problem = "一个两位数，十位比个位大 3，两数之和是 9。这个数是多少？只回答数字。"

attempt1 = chat(client, problem, max_tokens=50)
print("第 1 次尝试:", attempt1)

# 裁判（规则）：正确答案是 63
correct = "63" in attempt1
print("判分:", "✅" if correct else "❌")

if not correct:
    reflection = chat(client, f"题目：{problem}\n你的错误答案：{attempt1}\n写一句反思：你犯了什么错？下次如何避免？",
                      system="你是善于复盘的解题者，反思要具体可操作。", max_tokens=150)
    print("\n反思:", reflection)
    attempt2 = chat(client, f"{problem}\n\n上次教训：{reflection}", max_tokens=50)
    print("\n第 2 次尝试（带反思）:", attempt2, "→", "✅" if "63" in attempt2 else "❌")


第 1 次尝试: 
判分: ❌



反思: 



第 2 次尝试（带反思）:  → ❌


## 练习

1. 修改 `ReflexionLoop`：让反思不只基于单次失败，而是综合过去所有相关 episode 来生成更深层的洞察。
2. 实现一个 `retrieve_relevant(reflections, current_task)` 函数：用 TF-IDF 找到与当前任务最相关的历史教训。
3. 思考 Reflexion 的风险：如果反思本身是错误的（如归因错误），会导致什么后果？如何缓解？

> **检查点**：能用 Reflexion 模式为你自己的 Agent 项目设计一个「自我改进」机制。